
# ProofLaunch Cortex-X V10 Radeon Gate

**Track 1 — Development of Multimodal Content Creation Tools**

Run only when the kernel shows **Connected**. This gate fails closed. It does
not publish, submit a PR, overwrite the Gemma training Notebook, or reinterpret
the procedural fallback as model output.


In [ ]:

from pathlib import Path
import hashlib, json, os, platform, subprocess, sys, time

print("python:", sys.version)
print("platform:", platform.platform())
print("cwd:", Path.cwd())


## 1. Reassemble, hash-verify, and extract the versioned source


In [ ]:

archive = Path("prooflaunch-radeon-v10.tar.gz")
parts = sorted(Path(".").glob("prooflaunch-radeon-v10.part-*"))
if not archive.exists() and parts:
    with archive.open("wb") as destination:
        for part in parts:
            destination.write(part.read_bytes())
    print(f"reassembled {len(parts)} parts -> {archive}")

checksum_file = Path("prooflaunch-radeon-v10.sha256")
assert archive.is_file(), f"Missing archive and parts: {archive}"
assert checksum_file.is_file(), f"Missing checksum: {checksum_file}"
expected = checksum_file.read_text().split()[0].strip()
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print("archive sha256:", actual)
assert actual == expected, f"Archive hash mismatch: expected {expected}, got {actual}"
subprocess.run(["tar", "-xzf", str(archive)], check=True)
repo = Path("prooflaunch_submission")
assert (repo / "pyproject.toml").is_file(), "Extracted repository is incomplete"


## 2. Install declared dependencies and run Level-1 gates


In [ ]:

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo}[radeon]"],
    check=True,
)
env = {**os.environ, "PYTHONPATH": str(repo / "src")}
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(repo / "tests"), "-v"],
    env=env, text=True, capture_output=True
)
print(tests.stdout)
print(tests.stderr)
assert tests.returncode == 0, "Unit/integration tests failed"

control_eval = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts" / "run_control_plane_eval.py"),
        "--cases-per-mode", "50",
        "--output", str(repo / "evidence" / "radeon" / "control-plane-eval.json"),
    ],
    env=env, text=True, capture_output=True
)
print(control_eval.stdout)
print(control_eval.stderr)
assert control_eval.returncode == 0, "250-case control-plane gate failed"


## 3. Fail closed on the local AMD Radeon runtime

Before any heavy model is loaded, V7.4 writes a machine-readable
`ProofLaunchRadeonPreflightV2` report. The gate checks ROCm/HIP, VRAM, free
disk, every required Python dependency, executable FFmpeg, the local Gemma
base plus safe LoRA and frozen multimodal-retention evidence, all seven
external Hugging Face specialists against a release-locked repository,
40-character commit, license record, and exact cached revision, and—when
present—the exact 30 × 3 × 5 manifest plus every product and baseline source
hash. Processor and weight loaders receive the same immutable revision.

A missing cache is advisory while downloads are allowed and blocking under
`local_files_only` or `--strict-cache`. All other failed requirements block
the expensive matrix before it can produce misleading partial evidence.


In [ ]:
import torch

runtime = {
    "torch_version": torch.__version__,
    "hip_version": getattr(torch.version, "hip", None),
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "total_vram_bytes": (
        torch.cuda.get_device_properties(0).total_memory
        if torch.cuda.is_available() else 0
    ),
}
print(json.dumps(runtime, indent=2))
assert runtime["gpu_available"], "AMD Radeon GPU is not available"
assert runtime["hip_version"], "PyTorch does not report a ROCm/HIP runtime"

from prooflaunch_remix.model_registry import RadeonModelRouter
from prooflaunch_remix.radeon_preflight import (
    PREFLIGHT_SCHEMA,
    run_radeon_preflight,
    write_preflight_report,
)

available_vram_gb = runtime["total_vram_bytes"] / (1024 ** 3)
execution_plan = RadeonModelRouter().report(
    available_vram_gb=available_vram_gb,
    roles=(
        "vision_language_audit",
        "open_set_detection",
        "affect_critic",
        "image_generation",
        "image_text_scoring",
        "video_generation",
        "temporal_segmentation",
        "audio_text_scoring",
    ),
    include_candidates=True,
)
print(json.dumps(execution_plan, indent=2))

benchmark_dir = multimodal_run_dir / "competition-2x"
manifest_candidate = benchmark_dir / "manifest.json"
preflight_report = run_radeon_preflight(
    settings_path=repo / "config" / "radeon_matrix_adapter.json",
    manifest_path=manifest_candidate if manifest_candidate.exists() else None,
    strict_cache=False,
    minimum_vram_gb=16.0,
    minimum_free_disk_gb=40.0,
)
preflight_output = repo / "evidence" / "radeon" / "radeon-preflight.json"
write_preflight_report(preflight_report, preflight_output)
print(json.dumps(preflight_report, indent=2, sort_keys=True))
assert preflight_report["schema"] == PREFLIGHT_SCHEMA
assert preflight_report["passed"], {
    check["check_id"]: check["details"]
    for check in preflight_report["checks"]
    if check["status"] == "blocked"
}



## 3A. Build the inspectable multimodal mission before loading any model

This V7 path accepts independently hashed image, PDF, audio, and video inputs.
The frozen demonstration uses an owned product image plus an exact-page PDF
brief, then requests both a social image and a short image-conditioned video.
No free-running swarm may change the graph after its hash is recorded.


In [ ]:

import gc
from PIL import Image, ImageDraw
import fitz

from prooflaunch_remix.mission import (
    DeliverableKind,
    DeliverableSpec,
    MultimodalMissionPlanner,
)
from prooflaunch_remix.quality_contract import QualityTier
from prooflaunch_remix.multimodal_ingest import (
    MediaSource,
    MultimodalIngestor,
    RightsStatus,
)

multimodal_run_dir = repo / "evidence" / "radeon" / "multimodal-v7"
multimodal_run_dir.mkdir(parents=True, exist_ok=True)
mission_product_path = multimodal_run_dir / "owned-product.png"
mission_product = Image.new("RGBA", (720, 900), (0, 0, 0, 0))
mission_draw = ImageDraw.Draw(mission_product)
mission_draw.rounded_rectangle(
    (95, 55, 625, 835), radius=90,
    fill=(245, 248, 255, 255), outline=(174, 201, 239, 255), width=12,
)
mission_draw.rounded_rectangle(
    (185, 180, 535, 565), radius=36, fill=(16, 74, 154, 255)
)
mission_draw.ellipse((275, 655, 445, 825), fill=(255, 187, 76, 255))
mission_product.save(mission_product_path, format="PNG", optimize=False)

brief_pdf_path = multimodal_run_dir / "owned-brief.pdf"
brief_document = fitz.open()
brief_page = brief_document.new_page()
brief_page.insert_text(
    (72, 96),
    "Aster Arc evidence brief\nBattery life up to 18 hours under the documented test.\n"
    "Market: Japan. Platform: TikTok. Target emotion: joy.",
)
brief_document.save(brief_pdf_path)
brief_document.close()

mission_artifacts = MultimodalIngestor().ingest(
    (
        MediaSource(
            str(mission_product_path),
            "product_identity",
            RightsStatus.OWNED,
        ),
        MediaSource(
            str(brief_pdf_path),
            "evidence_brief",
            RightsStatus.OWNED,
        ),
    )
)
mission_plan = MultimodalMissionPlanner().build(
    mission_id="radeon-multimodal-v7",
    artifacts=mission_artifacts,
    deliverables=(
        DeliverableSpec(DeliverableKind.SOCIAL_IMAGE, "9:16"),
        DeliverableSpec(DeliverableKind.SHORT_VIDEO, "3:2", 4),
    ),
    available_vram_gb=available_vram_gb,
    quality_tier=QualityTier.COMPETITION_2X,
)
mission_bundle = {
    "artifacts": [item.serializable() for item in mission_artifacts],
    "plan": mission_plan.serializable(),
    "graph_hash": mission_plan.graph_hash,
}
(multimodal_run_dir / "mission-plan.json").write_text(
    json.dumps(mission_bundle, indent=2), encoding="utf-8"
)
print(json.dumps(mission_bundle, indent=2))



## 3B. Understand image evidence with local Qwen3-VL; extract PDF text exactly

Qwen3-VL must return source IDs and spatial/frame locators. The PDF extractor
keeps exact page text rather than asking a model to paraphrase evidence. Both
adapters fail closed on missing locators or invented source IDs.


In [ ]:

from prooflaunch_remix.multimodal_models import (
    LocalPDFTextExtractor,
    LocalQwen3VLAnalyzer,
)
from prooflaunch_remix.multimodal_ingest import MediaKind

image_artifacts = tuple(
    item for item in mission_artifacts if item.kind is MediaKind.IMAGE
)
pdf_artifact = next(
    item for item in mission_artifacts if item.kind is MediaKind.PDF
)
qwen_analyzer = LocalQwen3VLAnalyzer(
    model_id="Qwen/Qwen3-VL-2B-Instruct",
    max_new_tokens=512,
)
qwen_understanding = qwen_analyzer.analyze(
    image_artifacts,
    mission_prompt=(
        "Describe only visible product identity, colors, shape, logo/text, "
        "and composition constraints for an evidence-grounded launch."
    ),
)
pdf_observations = LocalPDFTextExtractor().extract(pdf_artifact)
understanding_evidence = {
    "qwen": {
        "summary": qwen_understanding.summary,
        "observations": [item.__dict__ for item in qwen_understanding.observations],
        "model_ref": qwen_understanding.model_ref,
    },
    "pdf": [item.__dict__ for item in pdf_observations],
}
(multimodal_run_dir / "understanding.json").write_text(
    json.dumps(understanding_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(understanding_evidence, indent=2))

del qwen_analyzer
gc.collect()
torch.cuda.empty_cache()
print("Qwen3-VL phase complete; Radeon memory released")



## 3C. Ground the product with local GroundingDINO

This is a spatial specialist, not a second general chat model. It converts the
owned product phrase into a scored box and source-located observations. The
selected box is later mapped into the composited video and becomes SAM2's
tracking prompt. The model is unloaded before Gemma starts.


In [ ]:

from prooflaunch_remix.multimodal_models import LocalGroundingDINO

product_artifact = image_artifacts[0]
dino = LocalGroundingDINO(
    model_id="IDEA-Research/grounding-dino-tiny",
    box_threshold=0.30,
    text_threshold=0.25,
)
dino_detections = dino.detect(
    product_artifact,
    text_labels=("electronic product", "electronic device", "product"),
)
dino_detection = max(dino_detections, key=lambda item: item.score)
dino_observations = tuple(
    item.to_observation(index)
    for index, item in enumerate(dino_detections, start=1)
)
dino_evidence = {
    "schema": "ProofLaunchGroundingDINOEvidenceV1",
    "model_ref": dino.model_id,
    "selected": dino_detection.__dict__,
    "detections": [item.__dict__ for item in dino_detections],
    "observations": [item.__dict__ for item in dino_observations],
}
(multimodal_run_dir / "grounding-dino.json").write_text(
    json.dumps(dino_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(dino_evidence, indent=2))

del dino
gc.collect()
torch.cuda.empty_cache()
print("GroundingDINO phase complete; Radeon memory released")



## 4. Reload, evaluate, record, and unload the multimodal-safe Gemma adapter

The gate requires the multimodal-retention artifact produced by
`gemma4_emotion_lora_multimodal_safe_v2.ipynb`. It reloads the local adapter,
runs the frozen emotion set, records the campaign pulse, then unloads Gemma
before SDXL enters Radeon memory.


In [ ]:

import gc, re
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from prooflaunch_remix.audience_pulse import GemmaAudiencePulse, RecordedAudiencePulse
from prooflaunch_remix.contracts import CreativeBrief

base_model_dir = Path("./models/google/gemma-4-E4B-it")
adapter_dir = Path("./gemma4-it-emotion-lora-ms-single-gpu")
retention_path = adapter_dir / "multimodal_retention.json"
assert base_model_dir.is_dir(), f"Missing local Gemma model: {base_model_dir}"
assert adapter_dir.is_dir(), f"Missing safe LoRA adapter: {adapter_dir}"
assert retention_path.is_file(), (
    "Missing multimodal retention evidence; run the safe V2 training Notebook"
)
retention = json.loads(retention_path.read_text(encoding="utf-8"))
assert retention["post_accuracy"] >= retention["minimum_post_accuracy"]
assert (
    retention["pre_accuracy"] - retention["post_accuracy"]
    <= retention["maximum_allowed_drop"]
)

gemma_tokenizer = AutoTokenizer.from_pretrained(
    base_model_dir,
    use_fast=True,
    trust_remote_code=True,
)
gemma_base = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda")
gemma_ft = PeftModel.from_pretrained(gemma_base, adapter_dir).to("cuda")
gemma_ft.eval()
valid_emotions = {"sadness", "joy", "love", "anger", "fear", "surprise"}

def predict_emotion_ft(user_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Read the user's text and answer with exactly one label. "
                "Only choose from: sadness, joy, love, anger, fear, surprise."
            ),
        },
        {
            "role": "user",
            "content": f"Classify the emotion of this text:\n\n{user_text}",
        },
    ]
    inputs = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {key: value.to("cuda") for key, value in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = gemma_ft.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id,
            eos_token_id=gemma_tokenizer.eos_token_id,
        )
    raw = gemma_tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    ).strip().lower()
    match = re.search(r"\b(sadness|joy|love|anger|fear|surprise)\b", raw)
    return match.group(1) if match else "invalid"

audience_pulse = GemmaAudiencePulse(
    classifier=predict_emotion_ft,
    model_ref="google/gemma-4-E4B-it+emotion-lora:multimodal-safe-v2",
)
emotion_cases = [
    ("I feel joyful and bright today.", "joy"),
    ("This wonderful launch makes me happy.", "joy"),
    ("I love spending time with you.", "love"),
    ("My heart is full of affection.", "love"),
    ("I am frightened by the sudden noise.", "fear"),
    ("That shadow makes me afraid.", "fear"),
    ("I am furious about the mistake.", "anger"),
    ("The unfair result makes me angry.", "anger"),
    ("I feel deeply sad about the loss.", "sadness"),
    ("The goodbye left me sorrowful.", "sadness"),
    ("What an unexpected surprise!", "surprise"),
    ("I cannot believe that happened.", "surprise"),
]
emotion_predictions = [
    {"text": text, "expected": expected, "predicted": predict_emotion_ft(text).strip().lower()}
    for text, expected in emotion_cases
]
emotion_accuracy = sum(
    row["expected"] == row["predicted"] for row in emotion_predictions
) / len(emotion_predictions)
emotion_evidence = {
    "model_ref": audience_pulse.model_ref,
    "case_count": len(emotion_predictions),
    "accuracy": emotion_accuracy,
    "required_accuracy": 0.80,
    "predictions": emotion_predictions,
    "limitation": "Six synthetic labels do not measure campaign persuasion.",
}
radeon_evidence = repo / "evidence" / "radeon"
radeon_evidence.mkdir(parents=True, exist_ok=True)
(radeon_evidence / "post-finetune-evaluation.json").write_text(
    json.dumps(emotion_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(emotion_evidence, indent=2))
assert emotion_accuracy >= 0.80, "Post-fine-tuning accuracy is below the precommitted bar"

preflight_brief = CreativeBrief(
    id="BRF-001",
    headline="Aster Arc — Battery life up to 18 hours under the documented test.",
    claim_ids=("CLM-001",),
    visual_direction="evidence-led launch",
    target_emotion="joy",
)
recorded_audience = RecordedAudiencePulse(audience_pulse.review(preflight_brief))

del audience_pulse, predict_emotion_ft, gemma_ft, gemma_base, gemma_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Gemma phase complete; Radeon memory released before image generation")



## 5. Run the key local visual inference on Radeon

The default is SDXL Base through Diffusers. Override `PROOFLAUNCH_IMAGE_MODEL`
only with a locally runnable model whose identifier and license are recorded.
The model creates the background; ProofLaunch composites the immutable product
layer, audits four mechanical gates plus the SigLIP2 semantic gate, repairs at
most twice, then exports.


In [ ]:

from diffusers import AutoPipelineForText2Image
from PIL import Image, ImageDraw

from prooflaunch_remix.contracts import CampaignRequest, Stage
from prooflaunch_remix.files import sha256_file
from prooflaunch_remix.ledger import EvidenceLedger
from prooflaunch_remix.runtime import build_supervisor
from prooflaunch_remix.studio import DiffusersBackgroundProvider
from prooflaunch_remix.visual_semantics import HuggingFaceSigLIP2Scorer

model_id = os.environ.get(
    "PROOFLAUNCH_IMAGE_MODEL",
    "stabilityai/stable-diffusion-xl-base-1.0",
)
steps = int(os.environ.get("PROOFLAUNCH_IMAGE_STEPS", "20"))
pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")
background = DiffusersBackgroundProvider(
    pipe,
    model_ref=model_id,
    steps=steps,
    guidance_scale=4.0,
    release_after_render=True,
)
semantic_scorer = HuggingFaceSigLIP2Scorer(
    model_id="google/siglip2-base-patch16-224",
    minimum_positive_probability=0.34,
    minimum_margin=0.05,
)

run_label = os.environ.get("PROOFLAUNCH_RUN_LABEL", "frozen-visual-001")
run_dir = radeon_evidence / run_label
run_dir.mkdir(parents=True, exist_ok=True)
product_path = run_dir / "fixture-product.png"
product = Image.new("RGBA", (720, 900), (0, 0, 0, 0))
draw = ImageDraw.Draw(product)
draw.rounded_rectangle(
    (95, 55, 625, 835), radius=90,
    fill=(245, 248, 255, 255), outline=(174, 201, 239, 255), width=12,
)
draw.rounded_rectangle((185, 180, 535, 565), radius=36, fill=(16, 74, 154, 255))
draw.ellipse((275, 655, 445, 825), fill=(255, 187, 76, 255))
product.save(product_path, format="PNG", optimize=False)

request = CampaignRequest(
    product_name="Aster Arc",
    product_image=str(product_path),
    specification_text="Battery life up to 18 hours under the documented test.",
    market="Japan",
    platform="TikTok",
    goal="launch",
    target_emotion="joy",
)
supervisor = build_supervisor(
    background=background,
    audience=recorded_audience,
    seed=202607,
    canvas_overrides={"TikTok": (768, 1344)},
    unsafe_first_pass=True,
    semantic_scorer=semantic_scorer,
)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
state = supervisor.run(request, str(run_dir), run_id="radeon-frozen-visual-001")
torch.cuda.synchronize()
latency_seconds = time.perf_counter() - started
peak_vram_bytes = torch.cuda.max_memory_allocated()
semantic_scorer.release()

assert state.stage is Stage.DELIVER, state.blocked_reason
assert not state.release_blockers, state.release_blockers
assert state.repair_count <= 2
assert state.assets[0].metadata["semantic_score"]["passed"]
assert EvidenceLedger.verify(run_dir / "evidence-ledger.jsonl")
visual_evidence = {
    "schema": "ProofLaunchRadeonVisualRunV1",
    "runtime": runtime,
    "execution_plan": execution_plan,
    "model_id": model_id,
    "steps": steps,
    "seed": 202607,
    "canvas": [768, 1344],
    "stage": state.stage.value,
    "repair_count": state.repair_count,
    "latency_seconds": latency_seconds,
    "peak_vram_bytes": peak_vram_bytes,
    "asset_path": state.assets[0].path,
    "asset_sha256": sha256_file(state.assets[0].path),
    "product_source_sha256": sha256_file(product_path),
    "protected_layer_sha256": state.assets[0].metadata["product_layer_sha256"],
    "semantic_score": state.assets[0].metadata.get("semantic_score"),
    "backend_kind": state.assets[0].metadata["backend_kind"],
    "exports": [
        {"path": path, "sha256": sha256_file(path)} for path in state.exports
    ],
}
(radeon_evidence / "generation-benchmark.json").write_text(
    json.dumps(visual_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(visual_evidence, indent=2))



## 5B. Generate a short local video with official LTX-Video 2B on Radeon

The 2B distilled checkpoint is the 24 GB-safe required tier. A separately
declared LTX-2.3 22B quality tier may be benchmarked only when the instance has
at least 46 GB and its ROCm patch is recorded. This cell uses the official
Lightricks inference library and records the exact Git revision and checkpoint.
It then overlays the owned product layer after generation on every frame.


In [ ]:

import imageio.v2 as imageio
from PIL import Image, ImageDraw

ltx_root = repo / ".vendor" / "LTX-Video"
if not (ltx_root / ".git").is_dir():
    ltx_root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Lightricks/LTX-Video.git",
            str(ltx_root),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{ltx_root}[inference]"],
    check=True,
)
ltx_revision = subprocess.check_output(
    ["git", "-C", str(ltx_root), "rev-parse", "HEAD"],
    text=True,
).strip()
sys.path.insert(0, str(ltx_root))
from ltx_video.inference import InferenceConfig, infer
from prooflaunch_remix.video_generation import IdentityLockedVideoComposer

ltx_output_dir = multimodal_run_dir / "ltx-video"
ltx_output_dir.mkdir(parents=True, exist_ok=True)
before_videos = set(ltx_output_dir.glob("*.mp4"))
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
video_started = time.perf_counter()
infer(
    InferenceConfig(
        pipeline_config=str(
            ltx_root / "configs" / "ltxv-2b-0.9.8-distilled.yaml"
        ),
        prompt=(
            "A precise studio product shot. The camera makes a slow dolly-in. "
            "Soft blue and amber light moves across a clean dark background. "
            "The supplied product remains centered, rigid, and unchanged. "
            "No text, no extra objects, no logo mutation."
        ),
        output_path=str(ltx_output_dir),
        seed=202607,
        height=512,
        width=768,
        num_frames=97,
        frame_rate=24,
        offload_to_cpu=available_vram_gb < 30,
        conditioning_media_paths=[str(mission_product_path)],
        conditioning_strengths=[1.0],
        conditioning_start_frames=[0],
    )
)
torch.cuda.synchronize()
video_latency_seconds = time.perf_counter() - video_started
video_peak_vram_bytes = torch.cuda.max_memory_allocated()
new_videos = set(ltx_output_dir.glob("*.mp4")) - before_videos
assert new_videos, "LTX-Video did not create a new MP4"
raw_video_path = max(new_videos, key=lambda item: item.stat().st_mtime_ns)

reader = imageio.get_reader(raw_video_path)
raw_frames = tuple(Image.fromarray(frame).convert("RGBA") for frame in reader)
reader.close()
locked_frames, identity_report = IdentityLockedVideoComposer().compose(
    generated_frames=raw_frames,
    product_layer=Image.open(mission_product_path).convert("RGBA"),
    normalized_box=(0.32, 0.28, 0.68, 0.78),
)
assert identity_report.passed, identity_report
# Add a measurable visual launch cue outside the protected product box.
cued_frames = []
for frame_index, frame in enumerate(locked_frames):
    rendered = frame.copy()
    if 24 <= frame_index <= 31:
        cue = ImageDraw.Draw(rendered)
        radius = 20 + 3 * (frame_index - 24)
        cue.ellipse(
            (54 - radius, 54 - radius, 54 + radius, 54 + radius),
            outline=(255, 187, 76, 255), width=8,
        )
    cued_frames.append(rendered)
locked_frames = tuple(cued_frames)
locked_video_path = ltx_output_dir / "prooflaunch-identity-locked.mp4"
with imageio.get_writer(
    locked_video_path,
    fps=24,
    codec="libx264",
    quality=10,
) as writer:
    for frame in locked_frames:
        writer.append_data(frame.convert("RGB"))
assert locked_video_path.is_file() and locked_video_path.stat().st_size > 0

video_evidence = {
    "schema": "ProofLaunchRadeonVideoRunV2",
    "runtime": runtime,
    "model": "Lightricks/LTX-Video:ltxv-2b-0.9.8-distilled.safetensors",
    "model_license": "LTX-Video Open Weights License 1.0",
    "official_code_revision": ltx_revision,
    "seed": 202607,
    "frame_count": identity_report.frame_count,
    "fps": 24,
    "latency_seconds": video_latency_seconds,
    "peak_vram_bytes": video_peak_vram_bytes,
    "raw_video_path": str(raw_video_path),
    "raw_video_sha256": sha256_file(raw_video_path),
    "locked_video_path": str(locked_video_path),
    "locked_video_sha256": sha256_file(locked_video_path),
    "pre_encode_identity_report": identity_report.__dict__,
    "truth_boundary": (
        "The pre-encode gate is exact. The following V7 delivery cell must "
        "also pass decoded H.264 identity, SAM2, CLAP, and A/V alignment."
    ),
}
(multimodal_run_dir / "video-benchmark.json").write_text(
    json.dumps(video_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(video_evidence, indent=2))

gc.collect()
torch.cuda.empty_cache()



## 5C. Track the grounded product through every generated frame with SAM2

The GroundingDINO box is mapped through the product compositing transform and
used as SAM2's first-frame prompt. A run passes only when at least 95% of
expected frames are returned and at least 95% contain a non-empty product mask.
SAM2 is unloaded before the audio-text model is loaded.


In [ ]:

from prooflaunch_remix.multimodal_models import LocalSAM2VideoTracker

source_width, source_height = Image.open(mission_product_path).size
video_width, video_height = locked_frames[0].size
container_box = (
    0.32 * video_width,
    0.28 * video_height,
    0.68 * video_width,
    0.78 * video_height,
)
dx1, dy1, dx2, dy2 = dino_detection.box_xyxy
container_width = container_box[2] - container_box[0]
container_height = container_box[3] - container_box[1]
sam_prompt_box = (
    container_box[0] + dx1 / source_width * container_width,
    container_box[1] + dy1 / source_height * container_height,
    container_box[0] + dx2 / source_width * container_width,
    container_box[1] + dy2 / source_height * container_height,
)
sam_tracker = LocalSAM2VideoTracker(
    model_id="facebook/sam2.1-hiera-small",
    minimum_frame_coverage=0.95,
    minimum_nonempty_mask_rate=0.95,
)
sam_report = sam_tracker.track(
    frames=locked_frames,
    prompt_box_xyxy=sam_prompt_box,
    source_artifact_id=product_artifact.id,
)
assert sam_report.passed, sam_report
(multimodal_run_dir / "sam2-temporal-segmentation.json").write_text(
    json.dumps(sam_report, default=lambda item: item.__dict__, indent=2),
    encoding="utf-8",
)
print(json.dumps(sam_report, default=lambda item: item.__dict__, indent=2))

del sam_tracker
gc.collect()
torch.cuda.empty_cache()
print("SAM2 phase complete; Radeon memory released")



## 5D. Produce the final A/V MP4, then audit decoded identity and timing

This cell creates a deterministic owned launch chime, muxes it into the local
video, decodes the actual delivery MP4, and applies four independent checks:
H.264 product identity, visible cue detection, CLAP audio semantics, and
timestamp alignment. Planned timestamps alone cannot pass this gate.


In [ ]:

import math
import wave
import imageio_ffmpeg
import numpy as np

from prooflaunch_remix.media_quality import (
    AudioVideoAlignmentGate,
    EncodedVideoIdentityGate,
    SemanticEvent,
)
from prooflaunch_remix.multimodal_models import LocalCLAPScorer

fps = 24
sample_rate = 48_000
duration_seconds = len(locked_frames) / fps
sample_count = round(duration_seconds * sample_rate)
timeline = np.arange(sample_count, dtype=np.float32) / sample_rate
audio = np.zeros(sample_count, dtype=np.float32)
cue_start, cue_end = 1.0, 1.34
cue_mask = (timeline >= cue_start) & (timeline <= cue_end)
cue_time = timeline[cue_mask] - cue_start
envelope = np.sin(np.pi * cue_time / (cue_end - cue_start)) ** 2
audio[cue_mask] = 0.40 * envelope * (
    np.sin(2 * np.pi * 659.25 * cue_time)
    + 0.55 * np.sin(2 * np.pi * 987.77 * cue_time)
)
audio = np.clip(audio, -0.95, 0.95)
source_audio_path = ltx_output_dir / "owned-launch-chime.wav"
with wave.open(str(source_audio_path), "wb") as handle:
    handle.setnchannels(1)
    handle.setsampwidth(2)
    handle.setframerate(sample_rate)
    handle.writeframes((audio * 32767).astype("<i2").tobytes())

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
final_video_path = ltx_output_dir / "prooflaunch-final-av.mp4"
subprocess.run(
    [
        ffmpeg, "-y",
        "-i", str(locked_video_path),
        "-i", str(source_audio_path),
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        str(final_video_path),
    ],
    check=True,
    capture_output=True,
)
assert final_video_path.is_file() and final_video_path.stat().st_size > 0

encoded_identity = EncodedVideoIdentityGate(
    codec_tolerance=0.04,
    minimum_frame_coverage=0.95,
).audit(
    video_path=final_video_path,
    product_layer=Image.open(mission_product_path).convert("RGBA"),
    normalized_box=(0.32, 0.28, 0.68, 0.78),
    expected_frame_count=len(locked_frames),
)
assert encoded_identity.passed, encoded_identity

final_reader = imageio.get_reader(final_video_path)
decoded_frames = tuple(Image.fromarray(frame).convert("RGB") for frame in final_reader)
final_reader.close()
cue_frames = []
for frame_index, frame in enumerate(decoded_frames):
    crop = np.asarray(frame.crop((5, 5, 103, 103)))
    amber = (
        (crop[:, :, 0] > 180)
        & (crop[:, :, 1] > 100)
        & (crop[:, :, 2] < 150)
    )
    if int(amber.sum()) >= 35:
        cue_frames.append(frame_index)
assert cue_frames, "the encoded visual cue was not detected"
video_event = SemanticEvent(
    "launch chime",
    min(cue_frames) / fps,
    (max(cue_frames) + 1) / fps,
    1.0,
    f"decoded_frames:{min(cue_frames)}-{max(cue_frames)}",
)

decoded_audio_path = ltx_output_dir / "decoded-delivery-audio.wav"
subprocess.run(
    [
        ffmpeg, "-y", "-i", str(final_video_path),
        "-vn", "-ac", "1", "-ar", str(sample_rate),
        str(decoded_audio_path),
    ],
    check=True,
    capture_output=True,
)
with wave.open(str(decoded_audio_path), "rb") as handle:
    decoded_audio = np.frombuffer(
        handle.readframes(handle.getnframes()),
        dtype="<i2",
    ).astype(np.float32) / 32768.0
    decoded_audio_duration = handle.getnframes() / handle.getframerate()

clap = LocalCLAPScorer(
    model_id="laion/clap-htsat-fused",
    minimum_probability=0.45,
)
clap_score = clap.score_array(
    audio=decoded_audio,
    sampling_rate=sample_rate,
    expected_label="a bright electronic launch chime",
    candidate_labels=(
        "a bright electronic launch chime",
        "spoken human voice",
        "traffic noise",
        "silence",
    ),
)
assert clap_score.passed, clap_score
audio_event = SemanticEvent(
    "launch chime",
    cue_start,
    cue_end,
    clap_score.expected_probability,
    f"decoded_audio_seconds:{cue_start:.2f}-{cue_end:.2f}",
)
alignment_report = AudioVideoAlignmentGate(
    tolerance_seconds=0.50,
    duration_tolerance_seconds=0.25,
    minimum_event_coverage=0.95,
).audit(
    audio_events=(audio_event,),
    video_events=(video_event,),
    audio_duration_seconds=decoded_audio_duration,
    video_duration_seconds=len(decoded_frames) / fps,
)
assert alignment_report.passed, alignment_report

delivery_evidence = {
    "schema": "ProofLaunchEncodedDeliveryGateV1",
    "final_video_path": str(final_video_path),
    "final_video_sha256": sha256_file(final_video_path),
    "encoded_identity": encoded_identity.to_dict(),
    "clap": clap_score.__dict__,
    "audio_video_alignment": alignment_report.to_dict(),
}
(multimodal_run_dir / "encoded-delivery-gate.json").write_text(
    json.dumps(delivery_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(delivery_evidence, indent=2))

del clap
gc.collect()
torch.cuda.empty_cache()
print("CLAP phase complete; Radeon memory released")



## 5F. Cross-media release gate

Every deliverable claim must point to a Qwen bounding box/frame locator or an
exact PDF page. Unknown rights, missing source IDs, or absent locators block the
pack. Video and image outputs stay in review until a human approves release.


In [ ]:

from prooflaunch_remix.cross_media import (
    CrossMediaReleaseGate,
    DeliverableClaim,
)

all_observations = (
    tuple(qwen_understanding.observations)
    + tuple(pdf_observations)
    + tuple(dino_observations)
)
claim_observation = next(
    item for item in pdf_observations if "18 hours" in item.claim
)
cross_media_findings = CrossMediaReleaseGate().audit(
    artifacts=mission_artifacts,
    observations=all_observations,
    claims=(
        DeliverableClaim(
            deliverable_id="social-image",
            text="Battery life up to 18 hours under the documented test.",
            observation_ids=(claim_observation.id,),
        ),
        DeliverableClaim(
            deliverable_id="short-video",
            text="Battery life up to 18 hours under the documented test.",
            observation_ids=(claim_observation.id,),
        ),
    ),
)
assert not cross_media_findings, cross_media_findings
cross_media_evidence = {
    "schema": "ProofLaunchCrossMediaGateV1",
    "finding_count": len(cross_media_findings),
    "passed": not cross_media_findings,
    "human_release_required": True,
    "mission_graph_hash": mission_plan.graph_hash,
}
(multimodal_run_dir / "cross-media-gate.json").write_text(
    json.dumps(cross_media_evidence, indent=2), encoding="utf-8"
)
print(json.dumps(cross_media_evidence, indent=2))


## 5G. Compile the frozen 30 × 3 × 5 evidence and apply all 20 bars

One successful fixture is evidence of operability, not benchmark quality.
V7 refuses hand-entered percentages. Certification requires:

- an immutable 450-case manifest created from 30 permitted product hashes;
- exactly one ProofLaunch and one baseline JSONL row for every case;
- at least 30 full audio-video cases, one per product;
- five-seed perceptual diversity without product-identity drift; and
- randomized candidate-hash reviews from at least five blinded raters.

The V7.4 matrix runner executes nine phase-batched passes so exactly one heavy
model is resident at a time. It atomically checkpoints each activity, re-hashes
successful artifacts and sidecars on resume, revalidates frozen input bytes,
retries only declared transient faults twice, and emits explicit failure rows.
Optional immutable-index shards are merged only after exact disjoint coverage
is proven. The bundled production factory routes Qwen3-VL, GroundingDINO,
multimodal-safe Gemma 4, SDXL, SigLIP2, LTX-Video, SAM2, and CLAP locally.

Every result row binds the still, required video, and per-case gate report by
relative locator plus SHA-256. Append-only journals fsync each completed case
and resume from the first missing manifest ID. The compiler re-hashes every
artifact, resolves A/B hashes after review, derives every rate, and aborts on
missing/duplicate cases, manifest drift, non-blind rows, copied seeds, unknown
outputs, or artifact tampering. The templates are in `docs/`; no benchmark
result is prefilled.


In [ ]:
from prooflaunch_remix.competition_benchmark import (
    BaselineCaseJournal,
    COMPILED_EVIDENCE_SCHEMA,
    CompetitionCaseJournal,
    compile_competition_evidence,
)
from prooflaunch_remix.benchmark_runner import (
    PHASE_ORDER,
    RUN_CHECKPOINT_SCHEMA,
)
from prooflaunch_remix.radeon_matrix_adapter import create_adapter
from prooflaunch_remix.radeon_preflight import PREFLIGHT_SCHEMA
from prooflaunch_remix.quality_contract import QualityTier, contract_for

contract = contract_for(QualityTier.COMPETITION_2X)
assert len(contract.metrics) == 20
assert len(PHASE_ORDER) == 9
assert preflight_report["schema"] == PREFLIGHT_SCHEMA
assert preflight_report["passed"]
adapter_identity = create_adapter().identity()
assert adapter_identity["system_kind"] == "prooflaunch"
assert adapter_identity["uses_closed_online_api"] is False
assert adapter_identity["model_supply_chain_schema"] == "ProofLaunchModelSupplyChainV1"
assert len(adapter_identity["model_supply_chain"]) == 7
assert all(
    "@" in model_ref
    for role, model_ref in adapter_identity["model_refs"].items()
    if role != "gemma"
)

# A plausible hand-entered aggregate must fail because it has no case-level
# compiler provenance.
uncompiled_fixture = {
    "product_count": 30,
    "aspect_ratio_count": 3,
    "seeds_per_product": 5,
    "video_case_count": 30,
    "human_raters": 5,
    "repair_budget_max": 2,
    "unsupported_claim_count": 0,
    "locator_coverage": 1.0,
    "image_identity_pass_rate": 1.0,
    "preencode_video_identity_pass_rate": 1.0,
    "encoded_video_identity_pass_rate": 1.0,
    "audio_video_alignment_pass_rate": 1.0,
    "cross_media_pass_rate": 1.0,
    "completion_rate": 1.0,
    "crash_rate": 0.0,
    "seed_diversity_group_pass_rate": 1.0,
    "baseline_completion_rate_gap": 0.0,
    "publish_preferred_count": 150,
    "publish_comparison_count": 150,
}
assert not contract.evaluate(uncompiled_fixture).passed

benchmark_dir = multimodal_run_dir / "competition-2x"
manifest_path = benchmark_dir / "manifest.json"
results_path = benchmark_dir / "prooflaunch-results.jsonl"
baseline_path = benchmark_dir / "baseline-results.jsonl"
human_path = benchmark_dir / "human-comparisons.csv"
required_paths = (manifest_path, results_path, baseline_path, human_path)

if manifest_path.exists():
    proof_journal = CompetitionCaseJournal(manifest_path, results_path)
    baseline_journal = BaselineCaseJournal(manifest_path, baseline_path)
    run_checkpoints = sorted(benchmark_dir.glob("run-checkpoint*.json"))
    checkpoint_summaries = []
    for checkpoint_path in run_checkpoints:
        checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        assert checkpoint["schema"] == RUN_CHECKPOINT_SCHEMA
        checkpoint_summaries.append(
            {
                "path": str(checkpoint_path),
                "phase_states": checkpoint.get("phase_states", {}),
                "activity_count": len(checkpoint.get("activities", {})),
            }
        )
    print(
        "durable matrix progress:",
        {
            "prooflaunch_recorded": proof_journal.recorded_count,
            "prooflaunch_pending": len(proof_journal.pending_cases),
            "baseline_pending": len(baseline_journal.pending_cases),
            "runner_checkpoints": checkpoint_summaries,
        },
    )

if all(path.exists() for path in required_paths):
    compiled = compile_competition_evidence(
        manifest_path=manifest_path,
        results_path=results_path,
        baseline_results_path=baseline_path,
        human_comparisons_path=human_path,
    )
    assert compiled["schema"] == COMPILED_EVIDENCE_SCHEMA
    assert compiled["benchmark_case_count"] == 450
    assert compiled["video_case_count"] >= 30
    report = contract.evaluate(compiled)
    (benchmark_dir / "competition-2x-evidence.json").write_text(
        json.dumps(compiled, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    (benchmark_dir / "competition-2x-report.json").write_text(
        json.dumps(report.to_dict(), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    assert report.passed, report.to_dict()
    print("competition-2X V7 benchmark: PASS (20/20)")
else:
    missing = [str(path) for path in required_paths if not path.exists()]
    print("competition-2X V7 benchmark: BLOCKED")
    print("Missing measured inputs:", json.dumps(missing, indent=2))
    print(
        "Build the manifest, then run scripts/run_competition_2x_matrix.py "
        "with --adapter "
        "prooflaunch_remix.radeon_matrix_adapter:create_adapter; "
        "do not replace missing case-level evidence with aggregate percentages."
    )



## 6. Submission handoff

Do not change `SUBMISSION_GATE.json` to ready from this one fixture. Complete the
30-product / three-ratio / five-seed benchmark, blinded human comparison,
3–5 minute real-operation video, and final English PR checklist first.


## 6. Official-first external teacher and Agent route

Every external model or Agent inference call first uses the official AMD
Radeon Token Factory endpoint. If all public official models fail with an
availability, rate-limit, server, or response-contract error, an approved
Fireworks route may take over for training-data generation, teacher
distillation, evaluation, or development Agent work.

Credentials come only from environment variables. The receipt contains no
credential, prompt, or output. This external route is never eligible as local
Radeon execution or competition-core benchmark evidence.

The current Token Factory account is capped at $10 per Asia/Shanghai daily
window and 30 RPM. The client keeps 24 RPM headroom, a 4,096-token request
ceiling, six attempts per logical completion, and a $0.0001 settlement buffer.
KV cache hits are optional provider telemetry, not a competition requirement.


In [ ]:
import hashlib
import os

from prooflaunch_remix.inference_gateway import (
    InferenceGatewayCallError,
    InferenceScope,
    OpenAICompatibleThirdPartyClient,
    ResilientInferenceGateway,
    load_third_party_provider_specs,
)
from prooflaunch_remix.tokenfactory import (
    TOKEN_FACTORY_API_KEY_ENV,
    TokenFactoryClient,
    TokenFactoryConfigurationError,
    tokenfactory_policy,
)

online_policy = tokenfactory_policy()
assert online_policy["competition_core_path"] == "local_radeon_only"
assert online_policy["external_inference_priority"][0] == (
    "amd_radeon_token_factory"
)
assert online_policy["quota"]["daily_budget_usd"] == 10.0
assert online_policy["quota"]["provider_rpm_limit"] == 30
assert online_policy["quota"]["client_rpm_limit"] == 24
assert online_policy["quota"]["daily_reserve_usd"] == 0.0001
assert online_policy["quota"]["max_tokens_per_request"] == 4096
route_receipt_path = (
    repo / "evidence" / "radeon" / "tokenfactory-route-receipt.json"
)
route_receipt_path.parent.mkdir(parents=True, exist_ok=True)

if not os.environ.get(TOKEN_FACTORY_API_KEY_ENV):
    route_receipt = {
        "schema": "ProofLaunchResilientInferenceGatewayV1",
        "status": "blocked",
        "reason": f"missing {TOKEN_FACTORY_API_KEY_ENV}",
        "credential": {
            "source": TOKEN_FACTORY_API_KEY_ENV,
            "persisted": False,
            "redacted": True,
        },
        "competition_core_eligible": False,
    }
else:
    fallback_specs = load_third_party_provider_specs(
        repo / "config" / "fireworks-fallback.example.json"
    )
    gateway = ResilientInferenceGateway(
        official_client=TokenFactoryClient(),
        third_party_clients=tuple(
            OpenAICompatibleThirdPartyClient(spec)
            for spec in fallback_specs
        ),
    )
    try:
        route_result = gateway.complete(
            [
                {
                    "role": "user",
                    "content": (
                        "Return exactly TOKENFACTORY_ROUTE_OK. "
                        "Do not add any other text."
                    ),
                }
            ],
            scope=InferenceScope.EVALUATION,
            purpose="Radeon Notebook external-route contract probe",
            max_tokens=32,
            temperature=0,
        )
        route_receipt = {
            **route_result.receipt,
            "response_content_sha256": hashlib.sha256(
                route_result.content.encode("utf-8")
            ).hexdigest(),
            "response_content_length": len(route_result.content),
        }
    except (InferenceGatewayCallError, TokenFactoryConfigurationError) as error:
        route_receipt = getattr(
            error,
            "receipt",
            {
                "schema": "ProofLaunchResilientInferenceGatewayV1",
                "status": "blocked",
                "reason": type(error).__name__,
                "competition_core_eligible": False,
            },
        )

route_receipt_path.write_text(
    json.dumps(route_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(route_receipt, indent=2, sort_keys=True))


## 7. Cortex-X V10: exact 1500-item program and truth boundary

This gate does not convert an ambitious TODO list into claimed results. It
loads the canonical 0001–1500 program, verifies all 36 workstreams and their
dependency waves, and keeps every item open until all declared D/A/I/E and
T/P/X evidence locators are hash-bound.

V10 also enforces four non-negotiable boundaries before training:

1. exactly one Supervisor owns authoritative state;
2. simulated/proxy business outcomes are displayed only as predicted;
3. every breakthrough must pass technical, product, and effect gates; and
4. a failed bet may close with negative evidence but cannot enter production.


In [ ]:
from prooflaunch_remix.quality_contract import QualityTier, contract_for
from prooflaunch_remix.v10_program import CortexProgram
from prooflaunch_remix.v10_training import default_v10_training_plan

program_path = repo / "docs" / "CORTEX_X_V10_1500_TODO.md"
program = CortexProgram.from_files((program_path,))
assert len(program.todos) == 1500
assert program.todos[0].todo_id == "0001"
assert program.todos[-1].todo_id == "1500"
assert len(program.ordered_workstreams()) == 36

v10_contract = contract_for(QualityTier.CORTEX_X_V10)
assert len(v10_contract.metrics) == 28
assert not v10_contract.evaluate({
    "schema": "ProofLaunchCortexXV10CompiledEvidenceV1",
    "four_a_overall_mean": 5.0,
}).passed, "partial aggregate must fail closed"

training_plan = default_v10_training_plan()
assert len(training_plan.stages) == 6
assert training_plan.radeon_required
assert training_plan.multi_gpu_required
assert training_plan.minimum_world_size >= 2
assert training_plan.preferred_world_size >= 4
assert training_plan.preferred_strategy == "ddp"
assert training_plan.memory_fallback_strategy == "fsdp2"
assert training_plan.single_gpu_baseline_required
assert not training_plan.single_gpu_may_promote
assert training_plan.maximum_resident_heavy_models_per_rank == 1
assert training_plan.judge_human_label_minimum >= 100
print(json.dumps({
    "todo_count": len(program.todos),
    "workstream_count": len(program.ordered_workstreams()),
    "quality_bar_count": len(v10_contract.metrics),
    "training_stage_count": len(training_plan.stages),
    "program_summary": program.summary,
}, indent=2))


## 8. Frontier-20-lens teacher-corpus gate

All 1,500 tasks must pass the nine-stage source-bound development pipeline
before the corpus is admitted to implementation planning:

`Architect + Research → Critic → Implementer → Verifier + 4A Critic → Synthesis → Frontier Judge → Final Repair`.

The validator blocks truncated output, placeholders, inferred licenses,
score-gate gaming, missing prerequisites, secret patterns, incomplete strict
JSON, or any teacher attempt to mark a TODO complete. These records remain
private external teacher advisory; they do not count as local Radeon evidence
and do not enter model weights until the separate provider-output rights gate
is explicitly unblocked. Raw episodes are intentionally excluded from the
public submission archive; mount or generate the private corpus only inside an
authorized environment.


In [ ]:
import subprocess
import sys

teacher_corpus = repo / "data" / "tokenfactory_teacher_v10"
teacher_receipt = (
    repo / "evidence" / "v10" / "tokenfactory-corpus-validation.json"
)
if not teacher_corpus.is_dir():
    raise FileNotFoundError(
        "Frontier teacher corpus is absent; all nine stages remain blocked"
    )
subprocess.run(
    [
        sys.executable,
        str(repo / "scripts" / "validate_tokenfactory_corpus.py"),
        "--corpus-root",
        str(teacher_corpus),
        "--output",
        str(teacher_receipt),
    ],
    check=True,
)
teacher_validation = json.loads(
    teacher_receipt.read_text(encoding="utf-8")
)
assert teacher_validation["status"] == "passed"
assert teacher_validation["quality_protocol"] == "frontier-20-lens-v3"
assert all(
    count == 750
    for count in teacher_validation["stage_valid_counts"].values()
)
assert teacher_validation["competition_core_eligible"] is False
teacher_rights = json.loads(
    (
        repo / "evidence" / "v10" / "tokenfactory-output-rights.json"
    ).read_text(encoding="utf-8")
)
assert teacher_rights["status"] == "rights_review_required"
assert teacher_rights["training_eligible"] is False
assert teacher_rights["redistribution_eligible"] is False
print(json.dumps(teacher_validation, indent=2, sort_keys=True))


## 8A. Top-10 agency campaign-distillation gate

The frozen 2026 North-American cohort must contain all ten ranked reference
agencies, at least one fully source-bound 16-layer case for each, and zero raw
campaign assets or sealed-evaluation overlap. The separate 300-case admission
gate remains blocked until every agency has 30 distinct cases.

All 1,500 TODOs also receive strategy, craft, effectiveness, and synthesis
passes. The final synthesis is strict JSON, references only the frozen case
IDs, carries a common Top-10 context digest, and stays in the
`research_reference` partition.


In [ ]:
four_a_manifest = json.loads(
    (
        repo / "config" / "north_america_4a_research_cohort_2026.json"
    ).read_text(encoding="utf-8")
)
assert len(four_a_manifest["agencies"]) == 10
required_layers = set(four_a_manifest["distillation_layers"])
fully_distilled_agencies = {
    case["agency_id"]
    for case in four_a_manifest["seed_cases"]
    if set(case["layers"]) == required_layers
}
assert fully_distilled_agencies == {
    agency["agency_id"] for agency in four_a_manifest["agencies"]
}
assert four_a_manifest["admission"]["raw_campaign_assets_stored"] is False
assert four_a_manifest["admission"]["training"] is False

four_a_receipt = (
    repo / "evidence" / "v10" / "four-a-transfer-corpus-validation.json"
)
subprocess.run(
    [
        sys.executable,
        str(repo / "scripts" / "validate_four_a_transfer_corpus.py"),
        "--corpus-root",
        str(teacher_corpus),
        "--output",
        str(four_a_receipt),
    ],
    check=True,
)
four_a_validation = json.loads(
    four_a_receipt.read_text(encoding="utf-8")
)
assert four_a_validation["status"] == "passed"
assert four_a_validation["complete"] is True
assert all(
    count == 750
    for count in four_a_validation["valid_counts"].values()
)
assert four_a_validation["four_a_context_sha256"]
assert four_a_validation["competition_core_eligible"] is False
print(json.dumps(four_a_validation, indent=2, sort_keys=True))


## 8B. Official-floor plus overdelivery gate

The phrase “200% quality” is not treated as an invented numeric score. It is
implemented as two non-substitutable gates for every canonical TODO:

1. every applicable official requirement; and
2. an overdelivery profile covering source, positive/negative tests, stop and
   rollback, plus applicable research, training, multimodal, 4K, Radeon,
   blind-4A, and human evidence.

Teacher prose, API spend, self-scores, simulations, and local proxies cannot
satisfy external evidence facets.


In [ ]:
double_standard_path = (
    repo / "evidence" / "v10" / "double-standard-requirements.json"
)
if not double_standard_path.exists():
    raise FileNotFoundError(
        "V10 double-standard requirements are absent; run "
        "scripts/build_v10_excellence_requirements.py"
    )
double_standard = json.loads(
    double_standard_path.read_text(encoding="utf-8")
)
assert double_standard["schema"] == (
    "ProofLaunchV10DoubleStandardRequirementsV1"
)
assert double_standard["todo_count"] == 1500
assert len(double_standard["requirements"]) == 1500
assert [
    item["todo_id"] for item in double_standard["requirements"]
] == [f"{index:04d}" for index in range(1, 1501)]
assert all(
    item["official_floor"] and item["overdelivery_gate"]
    for item in double_standard["requirements"]
)
assert double_standard["coverage"] == {
    "todos_with_applicable_official_floor": 1500,
    "todos_with_direct_submission_floor": 100,
    "todos_with_overdelivery_gate": 1500,
    "todos_with_required_facets": 1500,
    "todos_with_multigpu_program_binding": 1500,
    "todos_requiring_direct_multigpu_observation": 1100,
    "proxy_external_evidence_allowed": False,
}
print(json.dumps({
    "todo_count": double_standard["todo_count"],
    "requirements_sha256": double_standard["requirements_sha256"],
    "coverage": double_standard["coverage"],
}, indent=2))

multigpu_plan_path = (
    repo / "evidence" / "v10" / "multigpu-execution-plan.json"
)
if not multigpu_plan_path.exists():
    raise FileNotFoundError(
        "V10 multi-GPU plan is absent; run "
        "scripts/build_v10_multigpu_plan.py"
    )
multigpu_plan = json.loads(
    multigpu_plan_path.read_text(encoding="utf-8")
)
assert multigpu_plan["schema"] == (
    "ProofLaunchV10MultiGpuExecutionPlanV1"
)
assert multigpu_plan["coverage"]["todo_count"] == 1500
assert (
    multigpu_plan["coverage"]["todos_with_program_multigpu_binding"]
    == 1500
)
assert (
    multigpu_plan["coverage"][
        "todos_requiring_direct_multigpu_observation"
    ]
    == 1100
)
assert (
    multigpu_plan["strategy_policy"]["forbidden_training_strategy"]
    == "device_map=auto"
)
assert all(
    item["current_status"] == "not_started"
    for item in multigpu_plan["plans"]
)
multigpu_contract_for_hash = dict(multigpu_plan)
multigpu_expected_contract_hash = multigpu_contract_for_hash.pop(
    "contract_sha256"
)
assert hashlib.sha256(
    json.dumps(
        multigpu_contract_for_hash,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest() == multigpu_expected_contract_hash
print(json.dumps({
    "todo_plan_sha256": multigpu_plan["todo_plan_sha256"],
    "contract_sha256": multigpu_plan["contract_sha256"],
    "coverage": multigpu_plan["coverage"],
}, indent=2))


## 9. Multi-GPU Radeon curriculum execution gate

The curriculum is strictly sequential:

`multimodal CPT → curriculum SFT → direct feedback → on-policy distillation → multi-objective RL → sparse adapter consolidation`.

Inside each stage, DDP is the default after per-rank memory-fit preflight;
FSDP2 is allowed only as the verified memory fallback. World size, global
batch, immutable revisions, rank shards, and checkpoint ownership are frozen.
The one-GPU run is required only as an equivalence baseline and may not
promote a multi-GPU claim.

The vision/audio towers and base weights remain frozen. Each stage records
every-rank receipts, checkpoints atomically, runs general multimodal retention
plus stage-specific gates, and rolls back on failure. Training cannot start
from an ad-hoc dataset: the corpus manifest must prove content-hash
decontamination, temporal holdout, source locator, and
owned/permissive/public-domain rights for every sample.


In [ ]:
from prooflaunch_remix.v10_training import (
    DataRight,
    DataSplit,
    TrainingCorpusManifest,
    TrainingSample,
)

corpus_path = repo / "evidence" / "v10" / "training" / "corpus-manifest.json"
if not corpus_path.exists():
    raise FileNotFoundError(
        "V10 training is blocked: create the rights-audited, decontaminated "
        f"corpus manifest at {corpus_path}"
    )

corpus_payload = json.loads(corpus_path.read_text(encoding="utf-8"))
corpus = TrainingCorpusManifest(
    samples=tuple(
        TrainingSample(
            **{
                **row,
                "source_right": DataRight(row["source_right"]),
                "split": DataSplit(row["split"]),
            }
        )
        for row in corpus_payload["samples"]
    ),
    temporal_holdout_start=corpus_payload["temporal_holdout_start"],
)
corpus_audit = corpus.audit()
print(json.dumps(corpus_audit.__dict__, indent=2, default=str))
assert corpus_audit.passed, corpus_audit.failures
assert preflight_report["passed"], "Radeon preflight must pass before training"

multigpu_preflight_path = (
    repo / "evidence" / "RADEON_MULTIGPU_PREFLIGHT_2026-07-30.json"
)
multigpu_preflight = json.loads(
    multigpu_preflight_path.read_text(encoding="utf-8")
)
assert multigpu_preflight["status"] == "passed", (
    "Direct training is blocked until a >=2-GPU Radeon/RCCL preflight passes"
)
assert (
    multigpu_preflight["observed"]["torchCudaDeviceCount"]
    >= training_plan.minimum_world_size
)


## 10. V10 measured-result compiler

No metric is prefilled. The final cell accepts only a compiled result with raw
human comparison counts, five frozen baselines, real Radeon coverage, all
twelve breakthrough bundles, at least three promoted bets, and the 28 V10
quality bars. Absence of those files is a truthful blocked state.


In [ ]:
v10_evidence_path = (
    repo / "evidence" / "v10" / "cortex-x-v10-compiled-evidence.json"
)
v10_report_path = repo / "evidence" / "v10" / "cortex-x-v10-quality-report.json"

if not v10_evidence_path.exists():
    raise FileNotFoundError(
        "V10 certification is blocked: measured compiled evidence is absent"
    )
v10_evidence = json.loads(v10_evidence_path.read_text(encoding="utf-8"))
v10_report = v10_contract.evaluate(v10_evidence)
v10_report_path.parent.mkdir(parents=True, exist_ok=True)
v10_report_path.write_text(
    json.dumps(v10_report.to_dict(), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(v10_report.to_dict(), indent=2, sort_keys=True))
assert v10_report.passed, "one or more Cortex-X V10 bars failed"
